# 01 — Text Processing Basics

**Goal**: Understand how raw text gets converted into numbers that ML models can process.

As a prompt engineer, you write *words*. But LLMs see *tokens* — numbers. This notebook bridges that gap.

## What You'll Learn
- Tokenization (word, subword, character)
- Stopwords & punctuation removal
- Stemming vs Lemmatization
- Bag of Words & TF-IDF
- N-grams

In [10]:
# # Difference between Lemmatization and Stemming:

# from nltk.stem import PorterStemmer
# stemmer = PorterStemmer()
# print("running", stemmer.stem("running"))  # Output: run
# print("studies", stemmer.stem("studies"))    # Output: jump

# import spacy
# nlp = spacy.load("en_core_web_sm")
# doc = nlp("running studies")
# for token in doc:
#     print(token.text, token.lemma_)  # Output: running run, studies study


In [ ]:
# First run: install if needed
# !pip install nltk spacy scikit-learn
# !python -m spacy download en_core_web_sm

import nltk  # Core NLP: tokenization, stopwords, stemming, lemmatization
import spacy  # Advanced NLP: POS tagging, lemmatization, dependency parsing
import numpy as np  # Numerical computing: arrays, matrix ops, linear algebra
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # Convert text to number vectors (BoW & TF-IDF)

# Download NLTK data (one time)
nltk.download('punkt_tab', quiet=True)  #Teaches NLTK how to split text into sentences and words correctly
nltk.download('stopwords', quiet=True)  #Teaches NLTK how to identify common English stopwords
nltk.download('wordnet', quiet=True)    #Teaches NLTK about word relationships and synonyms

# Load spaCy model
nlp = spacy.load('en_core_web_sm')
print("All imports OK")

# What spaCy is actually known for

# When people say "spaCy," they usually mean a fast industrial-strength NLP library that provides:

# Tokenization
# Lemmatization
# Part-of-Speech (POS) tagging
# Named Entity Recognition (NER)
# Dependency Parsing
# Text Classification
# Word Vectors

All imports OK


## 1. What is Tokenization?

Tokenization = splitting text into smaller pieces (tokens).

- **Word tokens**: `"I love AI"` → `["I", "love", "AI"]`
- **Subword tokens**: `"unhappiness"` → `["un", "happi", "ness"]` (used by modern LLMs)
- **Character tokens**: `"AI"` → `["A", "I"]`

In [ ]:
text = "The quick brown fox jumps over the lazy dog. The dog barked!"

# Word tokenization with NLTK
from nltk.tokenize import word_tokenize, sent_tokenize  # Split text into words and sentences

words = word_tokenize(text)
sentences = sent_tokenize(text)

print(f"Original: {text}")
print(f"Sentences: {sentences}")
print(f"Words: {words}")
print(f"Word count: {len(words)}")

In [ ]:
# SpaCy tokenization (more sophisticated)
doc = nlp(text)
print(f"{'Token':<12} {'Lemma':<12} {'POS':<8} {'Is Stopword':<12}")
print("-" * 44)
for token in doc:
    print(f"{token.text:<12} {token.lemma_:<12} {token.pos_:<8} {token.is_stop!s:<12}")

## 2. Stopwords Removal

Stopwords = common words that carry little meaning ("the", "a", "is", "and").

Models often remove them to focus on meaningful words. **But**: modern LLMs keep them because grammar matters!

In [ ]:
from nltk.corpus import stopwords  # Common words list to filter noise from text

stop_words = set(stopwords.words('english'))

words_no_stop = [w for w in words if w.lower() not in stop_words and w.isalpha()]

print(f"All words: {words}")
print(f"Without stopwords: {words_no_stop}")

## 3. Stemming vs Lemmatization

Both reduce words to their root form. Difference:

- **Stemming**: crude chop — `"running"` → `"run"`, `"studies"` → `"studi"`
- **Lemmatization**: proper root using vocabulary — `"running"` → `"run"`, `"studies"` → `"study"`

LLMs use subword tokenizers (like BPE) — they don't stem. But understanding this helps with traditional NLP.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer  # Reduce words to root forms for normalization

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words_to_test = ["running", "studies", "better", "happiness", "corpora"]

print(f"{'Word':<15} {'Stem':<15} {'Lemma':<15}")
print("-" * 45)
for w in words_to_test:
    print(f"{w:<15} {stemmer.stem(w):<15} {lemmatizer.lemmatize(w):<15}")

## 4. Bag of Words (BoW)

A simple way to convert text to numbers: count how many times each word appears.

**Limitation**: ignores word order and semantics. "Dog bites man" = "Man bites dog" in BoW.

In [ ]:
documents = [
    "I love machine learning",
    "I love deep learning and neural networks",
    "Machine learning is amazing"
]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(documents)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nBag of Words matrix:")
print(bow_matrix.toarray())

## 5. TF-IDF (Term Frequency — Inverse Document Frequency)

TF-IDF = "how important is a word to a document, given the whole collection?"

- **TF**: how often a word appears in a document
- **IDF**: how rare a word is across all documents (rare = more important)

This is the foundation of information retrieval — and RAG still uses similar ideas!

In [ ]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(documents)

print("Vocabulary:", tfidf.get_feature_names_out())
print("\nTF-IDF matrix:")
print(np.round(tfidf_matrix.toarray(), 3))

## 6. N-grams

N-grams = groups of N consecutive words. They capture **word order** (unlike BoW).

- Unigram: `"I"`, `"love"`, `"AI"`
- Bigram: `"I love"`, `"love AI"`
- Trigram: `"I love AI"`

In [ ]:
vectorizer_ngram = CountVectorizer(ngram_range=(1, 2))  # unigrams + bigrams
ngram_matrix = vectorizer_ngram.fit_transform(documents)

print("N-gram features:", vectorizer_ngram.get_feature_names_out())
print("\nMatrix:")
print(ngram_matrix.toarray())

## Key Takeaways for GenAI

1. **Tokenization** is the first step for EVERY LLM. Different models use different tokenizers.
2. **Modern LLMs use subword tokenizers** (BPE, SentencePiece) — not simple word splitting.
3. **Stopwords matter less** for embeddings, but LLMs need them for grammatical output.
4. **TF-IDF** is still useful for traditional retrieval — but embeddings (next notebook) are better.
5. **N-grams** capture local word patterns, but attention mechanisms (Transformers) do this much better.

**Next**: Learn how embeddings turn words into meaningful vectors →